In [1]:
# ==============================================================================
# NEXUS OptiML 2026 — Operation Insight
# Complete Multi-Table Pipeline + Deep Learning (PyTorch FastTabularResNet):
# - Fixed NumPy 2.x / Python 3.14 np.clip() compatibility across all 5 tables
# - Deep Multi-Table Null & Outlier Reconstruction (Products, Stores, Customers, Calendar)
# - Stage A: Strict Out-of-Time Validation (Months 1-19 -> 20-24) matching Kaggle LB
# - Stage B: Full 5-Fold GroupKFold(order_id) CV across:
#     1. Optuna-Tuned LightGBM (GBDT)
#     2. Randomized ExtraTrees LightGBM
#     3. Optuna-Tuned PyTorch Deep Learning (FastTabularResNet with Pairwise AUC Loss)
#     4. Polynomial ElasticNet SGD + In-Process Feature-Bagging Logistic Ensemble
# ==============================================================================

import os
import gc
import time
import warnings
import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler, PowerTransformer, PolynomialFeatures
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import roc_auc_score

import lightgbm as lgb
import optuna

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
LGB_THREADS = min(8, os.cpu_count() or 4)
torch.set_num_threads(LGB_THREADS)
t_start = time.time()
print(f"Running on Device: {DEVICE} | CPU Threads: {LGB_THREADS}")

# ==============================================================================
# 1. DATA LOADING
# ==============================================================================
DATA_DIR = "."  # Set to "/kaggle/input/nexus-opti-ml-2026-operation-insight" on Kaggle

stores_df = pd.read_csv(os.path.join(DATA_DIR, "stores.csv"))
calendar_df = pd.read_csv(os.path.join(DATA_DIR, "calendar.csv"))
customers_df = pd.read_csv(os.path.join(DATA_DIR, "customers.csv"))
products_df = pd.read_csv(os.path.join(DATA_DIR, "products.csv"))
test_df = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))
sub_df = pd.read_csv(os.path.join(DATA_DIR, "sample_submission.csv"))

train_path = os.path.join(DATA_DIR, "train.csv")
if os.path.exists(train_path):
    train_df = pd.read_csv(train_path)
    print(f"Loaded train.csv: {train_df.shape} | test.csv: {test_df.shape}")
else:
    raise FileNotFoundError("Please ensure train.csv is in DATA_DIR.")

# ==============================================================================
# 2. CANONICAL MAPPINGS & VECTORIZED RELATIONAL HELPERS
# ==============================================================================
REGION_MAP = {
    "brazilsoutheast": "Brazil-Southeast", "brazilsutheast": "Brazil-Southeast",
    "brazilsouth": "Brazil-South",
    "brazilnortheast": "Brazil-Northeast", "brazilnrtheast": "Brazil-Northeast",
    "brazilcentralwest": "Brazil-CentralWest", "brazilcetralwest": "Brazil-CentralWest",
    "brazilnorth": "Brazil-North",
    "uknorth": "UK-North", "uknrth": "UK-North",
    "uksouth": "UK-South", "uksuth": "UK-South",
    "ukmidlands": "UK-Midlands", "ukmilands": "UK-Midlands",
    "uswest": "US-West",
    "useast": "US-East", "usest": "US-East", "usast": "US-East",
    "uscentral": "US-Central", "uscetral": "US-Central",
    "ussouth": "US-South", "ussuth": "US-South",
    "westerneurope": "Western Europe",
    "southerneurope": "Southern Europe", "southereurope": "Southern Europe",
    "nordics": "Nordics", "norics": "Nordics",
    "asiapacific": "Asia Pacific", "asiapcific": "Asia Pacific",
    "ireland": "Ireland", "ireand": "Ireland",
    "otherinternational": "Other International", "otherintrnational": "Other International",
    "americas": "Americas", "amercas": "Americas",
    "myanmaryangon": "Myanmar-Yangon",
    "myanmarmandalay": "Myanmar-Mandalay", "myanmarandalay": "Myanmar-Mandalay",
    "myanmarnaypyitaw": "Myanmar-Naypyitaw", "myanmaraypyitaw": "Myanmar-Naypyitaw",
}

REGION_TZ_OFFSET = {
    "Brazil-Southeast": -3.0, "Brazil-South": -3.0, "Brazil-Northeast": -3.0,
    "Brazil-CentralWest": -4.0, "Brazil-North": -4.0,
    "UK-North": 0.0, "UK-South": 0.0, "UK-Midlands": 0.0, "Ireland": 0.0,
    "Western Europe": 1.0, "Southern Europe": 1.0, "Nordics": 1.0,
    "Myanmar-Yangon": 6.5, "Myanmar-Mandalay": 6.5, "Myanmar-Naypyitaw": 6.5,
    "Asia Pacific": 8.0, "Other International": 0.0,
    "Americas": -4.5, "US-East": -4.5, "US-Central": -5.5, "US-South": -5.5, "US-West": -7.5,
}

MIDNIGHT_UTC_TIMES = {
    "00:01:00", "03:01:00", "23:01:00", "04:01:00", "05:01:00",
    "06:01:00", "07:01:00", "08:01:00", "22:01:00", "17:31:00", "16:01:00"
}
SHIFTED_UTC_TIMES = MIDNIGHT_UTC_TIMES - {"00:01:00"}

CAT_MAP = {
    "homedecor": "home_decor", "homeecor": "home_decor",
    "kitchendining": "kitchen_dining",
    "sportsleisure": "sports_leisure", "sportseisure": "sports_leisure",
    "toysgifts": "toys_gifts", "toysifts": "toys_gifts",
    "fashionaccessories": "fashion_accessories", "fashionacessories": "fashion_accessories",
    "officesupplies": "office_supplies", "officeupplies": "office_supplies",
    "healthbeauty": "health_beauty",
    "technology": "technology", "technlogy": "technology",
    "electronicsaccessories": "electronics_accessories",
    "gardendiy": "garden_diy", "gardediy": "garden_diy",
    "furniture": "furniture", "furnture": "furniture",
    "foodbeverages": "food_beverages", "foodbeerages": "food_beverages",
}

def fast_mode_impute(df, group_cols, target_col, default_val):
    valid = df.dropna(subset=[target_col])
    counts = valid.groupby(group_cols + [target_col], as_index=False).size()
    modes = counts.sort_values("size", ascending=False).drop_duplicates(group_cols)
    mode_map = modes.set_index(group_cols)[target_col]
    idx = df.set_index(group_cols).index
    return df[target_col].fillna(pd.Series(idx.map(mode_map), index=df.index)).fillna(default_val)

# ==============================================================================
# 3. DEEP MULTI-TABLE NULL & OUTLIER RECONSTRUCTION PIPELINE (NUMPY 2.X SAFE)
# ==============================================================================
def build_multi_table_dataset(train, test, stores, customers, products, calendar):
    t0 = time.time()
    print("Executing deep multi-table null & outlier reconstruction across all 5 datasets...")
    n_train = len(train)
    train["is_test"] = np.int8(0)
    test["is_test"] = np.int8(1)
    test["returned"] = np.float32(np.nan)
    
    df = pd.concat([train, test], axis=0, ignore_index=True)
    df["orig_idx"] = np.arange(len(df), dtype=np.int32)
    
    # --------------------------------------------------------------------------
    # A. STORES.CSV: Structural Nulls (size_sqm) & Format-Relative Normalization
    # --------------------------------------------------------------------------
    stores = stores.copy()
    stores["region"] = (
        stores["region"].astype(str).str.lower()
        .str.replace(r"[^a-z0-9]", "", regex=True).map(REGION_MAP).fillna("Unknown")
    )
    stores["is_virtual_store"] = stores["size_sqm"].isna().astype(np.int8)
    fmt_med_size = stores.groupby("format")["size_sqm"].transform("median")
    stores["size_sqm_clean"] = stores["size_sqm"].fillna(fmt_med_size).fillna(0.0).astype(np.float32)
    stores["store_size_vs_format"] = np.where(
        stores["is_virtual_store"] == 1, 1.0, stores["size_sqm_clean"] / np.clip(fmt_med_size, 1.0, None)
    ).astype(np.float32)
    stores["manager_tenure_years"] = np.clip(stores["manager_tenure_years"], 0.0, 25.0).astype(np.float32)
    stores["seller_rating"] = stores["seller_rating"].astype(np.float32)
    stores.drop(columns=["opened_date", "size_sqm"], inplace=True)
    
    # --------------------------------------------------------------------------
    # B. PRODUCTS.CSV: Category Cleaning, Zero-Weight Outlier Fix & Density Imputation
    # --------------------------------------------------------------------------
    products = products.copy()
    products["category"] = (
        products["category"].astype(str).str.lower()
        .str.replace(r"[^a-z0-9]", "", regex=True).map(CAT_MAP).fillna("unknown")
    )
    for c in ["list_price", "length_cm", "width_cm", "height_cm", "review_count"]:
        products[c] = products[c].astype(np.float32)
        
    products["volume_cm3"] = (
        np.clip(products["length_cm"], 1.0, None)
        * np.clip(products["width_cm"], 1.0, None)
        * np.clip(products["height_cm"], 1.0, None)
    ).astype(np.float32)
    products["dim_weight_kg"] = (products["volume_cm3"] / 5000.0).astype(np.float32)
    products["aspect_ratio"] = (
        products[["length_cm", "width_cm", "height_cm"]].max(axis=1)
        / np.clip(products[["length_cm", "width_cm", "height_cm"]].min(axis=1), 1.0, None)
    ).astype(np.float32)
    
    products["weight_was_invalid"] = (products["weight_kg"].isna() | (products["weight_kg"] <= 0.01)).astype(np.int8)
    valid_w = products["weight_kg"] > 0.01
    products["density_g_cm3"] = np.where(valid_w, (products["weight_kg"] * 1000.0) / products["volume_cm3"], np.nan)
    
    d_sup = products.groupby(["category", "packaging_type", "supplier_code"])["density_g_cm3"].transform("median")
    d_cat = products.groupby(["category", "packaging_type"])["density_g_cm3"].transform("median")
    d_final = d_sup.fillna(d_cat).fillna(products["density_g_cm3"].median())
    
    products["weight_kg"] = np.clip(
        np.where(valid_w, products["weight_kg"], (products["volume_cm3"] * d_final) / 1000.0),
        0.02, 45.0
    ).astype(np.float32)
    products["density_g_cm3"] = ((products["weight_kg"] * 1000.0) / products["volume_cm3"]).astype(np.float32)
    products["bulky_dim_to_weight_ratio"] = (products["dim_weight_kg"] / np.clip(products["weight_kg"], 0.02, None)).astype(np.float32)
    
    cat_med_p = products.groupby("category")["list_price"].transform("median")
    cat_iqr_p = np.clip(
        products.groupby("category")["list_price"].transform(lambda s: s.quantile(0.75) - s.quantile(0.25)),
        0.5, None
    )
    cat_med_w = products.groupby("category")["weight_kg"].transform("median")
    products["prod_price_vs_cat_med"] = (products["list_price"] / np.clip(cat_med_p, 0.1, None)).astype(np.float32)
    products["prod_price_robust_z"] = np.clip((products["list_price"] - cat_med_p) / cat_iqr_p, -5.0, 10.0).astype(np.float32)
    products["prod_weight_vs_cat_med"] = (products["weight_kg"] / np.clip(cat_med_w, 0.02, None)).astype(np.float32)
    
    sup_med_p = products.groupby("supplier_code")["list_price"].transform("median")
    products["prod_price_vs_sup_med"] = (products["list_price"] / np.clip(sup_med_p, 0.1, None)).astype(np.float32)
    products["log_review_count"] = np.log1p(np.clip(products["review_count"], 0.0, None)).astype(np.float32)
    products.drop(columns=["launch_date"], inplace=True)
    
    # --------------------------------------------------------------------------
    # C. CUSTOMERS.CSV: Duplicate Snapshot Resolution + Cross-Table Age Imputation
    # --------------------------------------------------------------------------
    customers = customers.copy()
    customers["region"] = (
        customers["region"].astype(str).str.lower()
        .str.replace(r"[^a-z0-9]", "", regex=True).map(REGION_MAP).fillna("Unknown")
    )
    tier_map = {"bronze": 1, "silver": 2, "gold": 3, "platinum": 4}
    customers["loyalty_rank"] = customers["loyalty_tier"].map(tier_map).fillna(1).astype(np.int8)
    customers["record_updated_at"] = pd.to_datetime(customers["record_updated_at"])
    customers.sort_values(["customer_id", "record_updated_at"], inplace=True)
    
    customers["age_band"] = customers.groupby("customer_id", sort=False)["age_band"].ffill().bfill()
    dup_counts = customers["customer_id"].value_counts()
    first_rank = customers.drop_duplicates("customer_id", keep="first").set_index("customer_id")["loyalty_rank"]
    
    customers = customers.drop_duplicates("customer_id", keep="last").copy()
    customers["cust_record_count"] = customers["customer_id"].map(dup_counts).fillna(1).astype(np.int8)
    customers["loyalty_upgrade_delta"] = (
        customers["loyalty_rank"] - customers["customer_id"].map(first_rank).fillna(customers["loyalty_rank"])
    ).astype(np.int8)
    customers["newsletter_opt_in"] = customers["newsletter_opt_in"].astype(np.int8)
    customers["age_band_was_null"] = customers["age_band"].isna().astype(np.int8)
    
    cust_dom_dev = (
        df.groupby(["customer_id", "device_type"], as_index=False).size()
        .sort_values("size", ascending=False).drop_duplicates("customer_id")
        .set_index("customer_id")["device_type"]
    )
    customers["dom_device"] = customers["customer_id"].map(cust_dom_dev).fillna("web")
    customers["age_band"] = fast_mode_impute(
        customers, ["customer_type", "loyalty_tier", "preferred_contact", "dom_device"], "age_band", "25-34"
    )
    age_mid = {"18-24": 21, "25-34": 30, "35-44": 40, "45-54": 50, "55-64": 60, "65+": 70}
    customers["age_midpoint"] = customers["age_band"].map(age_mid).fillna(35).astype(np.float32)
    customers.drop(columns=["signup_date", "record_updated_at", "dom_device"], inplace=True)
    
    # --------------------------------------------------------------------------
    # D. CALENDAR.CSV: Stationary Seasonal Weather Bands & Fuel Surcharge
    # --------------------------------------------------------------------------
    calendar = calendar.copy()
    calendar["week_start"] = pd.to_datetime(calendar["week_start"])
    calendar.sort_values("week_start", inplace=True)
    calendar["holiday_flag"] = calendar["holiday_flag"].astype(np.int8)
    calendar["temp_regime"] = np.where(
        calendar["temperature_f"] < 48.0, 0, np.where(calendar["temperature_f"] <= 66.0, 1, 2)
    ).astype(np.int8)
    calendar["fuel_surcharge_cal"] = calendar["fuel_surcharge"].astype(np.float32)
    calendar_keep = calendar[["week_start", "holiday_flag", "temp_regime", "fuel_surcharge_cal"]].copy()
    
    # --------------------------------------------------------------------------
    # E. ORDERS (TRAIN/TEST): Within-Order Propagation & Multi-Table Merges
    # --------------------------------------------------------------------------
    for col in ["tz_hint", "payment_method", "ship_mode", "session_clicks"]:
        df[f"{col}_was_null"] = df[col].isna().astype(np.int8)
        
    df.sort_values("order_id", inplace=True, kind="mergesort")
    g_ord = df.groupby("order_id", sort=False)
    for col in ["tz_hint", "payment_method", "ship_mode", "session_clicks"]:
        df[col] = g_ord[col].ffill().bfill()
    df.sort_values("orig_idx", inplace=True, kind="mergesort")
    
    df["has_coupon"] = df["coupon_code"].notna().astype(np.int8)
    df["order_has_coupon"] = df.groupby("order_id", sort=False)["has_coupon"].transform("max").astype(np.int8)
    
    df = df.merge(products, on="product_id", how="left")
    df = df.merge(stores, on="store_id", how="left", suffixes=("", "_store"))
    df = df.merge(customers, on="customer_id", how="left", suffixes=("", "_cust"))
    
    df["order_dt"] = pd.to_datetime(df["order_ts"])
    df["week_start"] = df["order_dt"].dt.to_period("W-SUN").dt.start_time.clip(
        calendar_keep["week_start"].min(), calendar_keep["week_start"].max()
    )
    df = df.merge(calendar_keep, on="week_start", how="left")
    
    # --------------------------------------------------------------------------
    # F. SMART MULTI-TABLE IMPUTATION OF REMAINING TRANSACTION NULLS
    # --------------------------------------------------------------------------
    time_str = df["order_ts"].str[11:]
    df.loc[df["tz_hint"].isna() & time_str.isin(SHIFTED_UTC_TIMES), "tz_hint"] = "UTC"
    df.loc[df["tz_hint"].isna() & (df["channel"].isin(["catalog", "store"])) & (time_str == "00:01:00"), "tz_hint"] = "local"
    df["tz_hint"] = fast_mode_impute(df, ["region_cust", "channel", "device_type"], "tz_hint", "UTC")
    
    df.loc[df["ship_mode"].isna() & ((df["channel"] == "store") | (df["device_type"] == "pos")), "ship_mode"] = "in_store"
    df["ship_mode"] = fast_mode_impute(df, ["channel", "return_window_days", "format"], "ship_mode", "standard")
    
    df["payment_method"] = fast_mode_impute(
        df, ["channel", "device_type", "customer_type", "region_cust"], "payment_method", "card"
    )
    
    sc_med = df.groupby(["device_type", "channel", "customer_type"])["session_clicks"].transform("median")
    df["session_clicks"] = np.clip(df["session_clicks"].fillna(sc_med).fillna(9.0), 0.0, 28.0).astype(np.float32)
    
    # --------------------------------------------------------------------------
    # G. OUTLIER REPAIR & RELATIONAL IMPUTATION OF PRICING, DISCOUNT & FREIGHT
    # --------------------------------------------------------------------------
    for c in ["quantity", "order_value", "line_no", "items_in_order", "return_window_days", "banner_id"]:
        df[c] = df[c].astype(np.float32)
    df["gift_wrap"] = df["gift_wrap"].astype(np.int8)
    
    raw_line_total = df["unit_price"] * np.clip(df["quantity"], 1.0, None)
    cents_bug = (
        (df["unit_price"] / np.clip(df["list_price"], 0.01, None) > 8.0)
        | ((df["order_value"] > 0) & ((df["order_value"] / raw_line_total) < 0.05))
    )
    df["is_cents_bug"] = cents_bug.astype(np.int8)
    df["clean_unit_price"] = np.where(cents_bug, df["unit_price"] / 100.0, df["unit_price"]).astype(np.float32)
    df["clean_unit_price"] = np.minimum(df["clean_unit_price"], 5.0 * df["list_price"]).astype(np.float32)
    
    df["discount_was_null"] = df["discount"].isna().astype(np.int8)
    actual_price_disc = np.clip(1.0 - (df["clean_unit_price"] / np.clip(df["list_price"], 0.01, None)), 0.0, 0.80)
    implied_disc = np.clip(1.0 - (df["clean_unit_price"] / (0.963 * np.clip(df["list_price"], 0.01, None))), 0.0, 0.80)
    ord_med_disc = df.groupby("order_id", sort=False)["discount"].transform("median")
    blended_disc_fill = np.where(
        ord_med_disc.notna() & (implied_disc > 0.05),
        0.85 * implied_disc + 0.15 * ord_med_disc,
        implied_disc
    )
    df["discount"] = np.clip(df["discount"].fillna(pd.Series(blended_disc_fill, index=df.index)), 0.0, 0.80).astype(np.float32)
    
    df["overcharged_vs_discount"] = (df["discount"] - actual_price_disc).astype(np.float32)
    df["is_overcharged_promo"] = (df["overcharged_vs_discount"] > 0.12).astype(np.int8)
    
    df["expected_line_val"] = (df["clean_unit_price"] * df["quantity"]).astype(np.float32)
    df["order_val_was_zero"] = (df["order_value"] <= 0.01).astype(np.int8)
    df["clean_order_value"] = np.where(
        df["order_value"] <= 0.01, df["expected_line_val"], df["order_value"]
    ).astype(np.float32)
    df["billing_discrepancy_ratio"] = np.clip(
        df["clean_order_value"] / np.clip(df["expected_line_val"], 0.01, None), 0.1, 5.0
    ).astype(np.float32)
    
    df["freight_was_null"] = df["freight_cost"].isna().astype(np.int8)
    df["log_quantity"] = np.log1p(np.clip(df["quantity"], 0.0, None)).astype(np.float32)
    df.loc[df["freight_cost"].isna() & (df["ship_mode"] == "in_store"), "freight_cost"] = 0.0
    
    df["freight_per_log_qty"] = df["freight_cost"] / np.clip(df["log_quantity"], 0.69, None)
    ord_fr_rate = df.groupby("order_id", sort=False)["freight_per_log_qty"].transform("median")
    cust_fr_rate = df.groupby("customer_id", sort=False)["freight_per_log_qty"].transform("median")
    prod_fr_rate = df.groupby("product_id", sort=False)["freight_per_log_qty"].transform("median")
    chan_ship_rate = df.groupby(["channel", "ship_mode"], sort=False)["freight_per_log_qty"].transform("median")
    
    est_fr_rate = (
        0.45 * ord_fr_rate.fillna(cust_fr_rate).fillna(chan_ship_rate)
        + 0.30 * prod_fr_rate.fillna(chan_ship_rate)
        + 0.25 * cust_fr_rate.fillna(chan_ship_rate)
    ).fillna(chan_ship_rate)
    
    est_freight = np.clip(est_fr_rate * df["log_quantity"], 0.0, 150.0)
    df["freight_cost"] = df["freight_cost"].fillna(est_freight).fillna(9.40).astype(np.float32)
    df["freight_vs_expected_ratio"] = np.clip(df["freight_cost"] / (est_freight + 1.0), 0.0, 5.0).astype(np.float32)
    df["fuel_deflated_freight"] = (df["freight_cost"] / (1.0 + df["fuel_surcharge_cal"].fillna(0.30))).astype(np.float32)
    df.drop(columns=["freight_per_log_qty", "fuel_surcharge_cal"], inplace=True)
    
    # --------------------------------------------------------------------------
    # H. CONTEXTUAL OUTLIER NORMALIZATION (Z-SCORES BY CUSTOMER_TYPE & CATEGORY)
    # --------------------------------------------------------------------------
    df["log_order_value"] = np.log1p(np.clip(df["clean_order_value"], 0.0, None)).astype(np.float32)
    df["log_items_in_order"] = np.log1p(np.clip(df["items_in_order"], 1.0, None)).astype(np.float32)
    
    grp_cc = df.groupby(["customer_type", "category"], sort=False)
    cc_med_val = grp_cc["log_order_value"].transform("median")
    cc_std_val = np.clip(grp_cc["log_order_value"].transform("std").fillna(1.0), 0.2, None)
    df["order_val_z_cust_cat"] = np.clip((df["log_order_value"] - cc_med_val) / cc_std_val, -4.0, 6.0).astype(np.float32)
    
    cc_med_qty = grp_cc["log_quantity"].transform("median")
    cc_std_qty = np.clip(grp_cc["log_quantity"].transform("std").fillna(1.0), 0.2, None)
    df["qty_z_cust_cat"] = np.clip((df["log_quantity"] - cc_med_qty) / cc_std_qty, -4.0, 6.0).astype(np.float32)
    
    # --------------------------------------------------------------------------
    # I. TIMEZONE RECONSTRUCTION, BASKET DYNAMICS & ENTITY SIGNATURES
    # --------------------------------------------------------------------------
    df["is_date_only_ts"] = time_str.isin(MIDNIGHT_UTC_TIMES).astype(np.int8)
    tz_offset_hrs = df["region_cust"].map(REGION_TZ_OFFSET).fillna(0.0)
    applied_shift_mins = np.where(df["tz_hint"] == "UTC", tz_offset_hrs * 60.0, 0.0)
    local_dt = df["order_dt"] + pd.to_timedelta(applied_shift_mins, unit="m")
    
    raw_local_hour = local_dt.dt.hour
    df["true_local_hour"] = np.where(df["is_date_only_ts"] == 1, -1, raw_local_hour).astype(np.int8)
    df["local_dayofweek"] = local_dt.dt.dayofweek.astype(np.int8)
    df["is_weekend"] = (df["local_dayofweek"] >= 5).astype(np.int8)
    df["hour_sin"] = np.where(df["is_date_only_ts"] == 1, 0.0, np.sin(2 * np.pi * raw_local_hour / 24.0)).astype(np.float32)
    df["hour_cos"] = np.where(df["is_date_only_ts"] == 1, 0.0, np.cos(2 * np.pi * raw_local_hour / 24.0)).astype(np.float32)
    
    df["line_weight_kg"] = (df["weight_kg"] * df["quantity"]).astype(np.float32)
    df["freight_to_order_val"] = np.clip(df["freight_cost"] / (df["clean_order_value"] + 1.0), 0.0, 5.0).astype(np.float32)
    df["unit_to_list_ratio"] = np.clip(df["clean_unit_price"] / np.clip(df["list_price"], 0.01, None), 0.0, 5.0).astype(np.float32)
    df["savings_dollars"] = np.log1p(np.clip(df["list_price"] - df["clean_unit_price"], 0.0, None) * df["quantity"]).astype(np.float32)
    
    g = df.groupby("order_id", sort=False)
    order_mean_spend = g["clean_order_value"].transform("mean").astype(np.float32)
    order_max_spend = g["clean_order_value"].transform("max").astype(np.float32)
    order_mean_qty = g["quantity"].transform("mean").astype(np.float32)
    order_mean_disc = g["discount"].transform("mean").astype(np.float32)
    order_max_disc = g["discount"].transform("max").astype(np.float32)
    
    df["order_mean_spend"] = np.log1p(order_mean_spend).astype(np.float32)
    df["order_mean_disc"] = order_mean_disc
    df["order_max_disc"] = order_max_disc
    df["line_vs_order_mean_spend"] = np.clip(df["clean_order_value"] / np.clip(order_mean_spend, 0.1, None), 0.0, 10.0).astype(np.float32)
    df["line_vs_order_max_spend"] = np.clip(df["clean_order_value"] / np.clip(order_max_spend, 0.1, None), 0.0, 1.0).astype(np.float32)
    df["line_vs_order_mean_qty"] = np.clip(df["quantity"] / np.clip(order_mean_qty, 0.1, None), 0.0, 10.0).astype(np.float32)
    df["line_disc_vs_order_mean"] = (df["discount"] - order_mean_disc).astype(np.float32)
    df["line_no_ratio"] = (df["line_no"] / np.clip(df["items_in_order"], 1.0, None)).astype(np.float32)
    df["clicks_per_log_item"] = (df["session_clicks"] / np.clip(df["log_items_in_order"], 0.69, None)).astype(np.float32)
    
    same_prod_cnt = df.groupby(["order_id", "product_id"], sort=False)["order_line_id"].transform("count").astype(np.float32)
    same_cat_cnt = df.groupby(["order_id", "category"], sort=False)["order_line_id"].transform("count").astype(np.float32)
    df["same_prod_in_order"] = np.log1p(same_prod_cnt).astype(np.float32)
    df["same_cat_share_in_order"] = (same_cat_cnt / np.clip(df["items_in_order"], 1.0, None)).astype(np.float32)
    df["is_bracket_cat_order"] = (same_cat_cnt > 1).astype(np.int8)
    
    df["is_same_region"] = (df["region_cust"] == df["region"]).astype(np.int8)
    df["cust_country"] = df["region_cust"].str.split("-").str[0]
    df["store_country"] = df["region"].str.split("-").str[0]
    df["is_same_country"] = (df["cust_country"] == df["store_country"]).astype(np.int8)
    df["is_same_channel"] = (df["channel"] == df["channel_store"]).astype(np.int8)
    
    for ent, prefix in [("product_id", "prod"), ("store_id", "store"), ("customer_id", "cust")]:
        ge = df.groupby(ent, sort=False)
        mean_val = ge["clean_order_value"].transform("mean").astype(np.float32)
        df[f"{prefix}_log_mean_val"] = np.log1p(mean_val).astype(np.float32)
        df[f"val_vs_{prefix}_mean"] = np.clip(df["clean_order_value"] / np.clip(mean_val, 0.1, None), 0.0, 10.0).astype(np.float32)
        df[f"{prefix}_mean_disc"] = ge["discount"].transform("mean").astype(np.float32)
        df[f"disc_vs_{prefix}_mean"] = (df["discount"] - df[f"{prefix}_mean_disc"]).astype(np.float32)
        df[f"{prefix}_gift_rate"] = ge["gift_wrap"].transform("mean").astype(np.float32)
        df[f"{prefix}_mean_ret_win"] = ge["return_window_days"].transform("mean").astype(np.float32)
        
    cust_code = pd.factorize(df["customer_id"])[0].astype(np.int64)
    prod_code = pd.factorize(df["product_id"])[0].astype(np.int64)
    store_code = pd.factorize(df["store_id"])[0].astype(np.int64)
    cat_code = pd.factorize(df["category"])[0].astype(np.int64)
    chan_code = pd.factorize(df["channel"])[0].astype(np.int64)
    ship_code = pd.factorize(df["ship_mode"])[0].astype(np.int64)
    zone_code = pd.factorize(df["warehouse_zone"])[0].astype(np.int64)
    temp_code = (df["temp_regime"] + 1).astype(np.int64)
    
    df["cust_cat_key"] = cust_code * 32 + cat_code
    df["prod_chan_key"] = prod_code * 16 + chan_code
    df["store_cat_key"] = store_code * 32 + cat_code
    df["cat_retwin_key"] = cat_code * 100 + df["return_window_days"].astype(np.int64)
    df["ship_zone_key"] = ship_code * 32 + zone_code
    df["cat_temp_key"] = cat_code * 8 + temp_code
    
    df.sort_values("orig_idx", inplace=True, kind="mergesort")
    print(f"Multi-table reconstruction & feature engineering completed in {time.time() - t0:.1f}s!")
    
    tr_out = df.iloc[:n_train].copy().reset_index(drop=True)
    te_out = df.iloc[n_train:].copy().reset_index(drop=True)
    return tr_out, te_out

train_fe, test_fe = build_multi_table_dataset(
    train_df, test_df, stores_df, customers_df, products_df, calendar_df
)
del train_df, test_df, stores_df, customers_df, products_df, calendar_df
gc.collect()

# ==============================================================================
# 4. DUAL-STAGE TARGET ENCODING (OUT-OF-TIME + 5-FOLD GROUPKFOLD)
# ==============================================================================
y_all = train_fe["returned"].astype(np.int8).values
train_order_id = train_fe["order_id"].values
test_order_id = test_fe["order_id"].values
test_order_line_id = test_fe["order_line_id"].values

order_time_rank = train_fe.groupby("order_id", sort=False)["order_dt"].transform("min").rank(method="first", pct=True).values
is_oot_val = (order_time_rank >= 0.80)
oot_tr_idx = np.where(~is_oot_val)[0]
oot_va_idx = np.where(is_oot_val)[0]

recency_weights = (0.6 + 0.8 * order_time_rank).astype(np.float32)
global_mean = float(y_all.mean())

N_SPLITS = 5
gkf = GroupKFold(n_splits=N_SPLITS)
folds = list(gkf.split(train_fe, y_all, groups=train_order_id))

HIER_TE_SPECS = [
    ("product_id", "category", 30.0),
    ("store_id", "format", 30.0),
    ("supplier_code", "category", 35.0),
    ("prod_chan_key", "channel", 30.0),
    ("store_cat_key", "category", 30.0),
    ("cat_retwin_key", "category", 40.0),
    ("ship_zone_key", "ship_mode", 40.0),
    ("cat_temp_key", "category", 40.0),
    ("banner_id", "channel", 40.0),
    ("customer_id", "customer_type", 50.0),
    ("cust_cat_key", "category", 35.0),
]

print("\n--- Computing Leak-Free Out-of-Time TE (for Local LB Alignment) & 5-Fold OOF TE ---")
oot_val_te_matrix = {}
te_col_names = []

for col, prior_col, m in HIER_TE_SPECS:
    fname = f"{col}_eb_te"
    te_col_names.append(fname)
    
    # 1. Strict Out-of-Time TE (fitted ONLY on Months 1-19, mapped to Months 20-24)
    tr_oot_sub = train_fe.iloc[oot_tr_idx]
    p_oot = tr_oot_sub.groupby(prior_col)["returned"].mean()
    s_oot = tr_oot_sub.groupby(col)["returned"].agg(["count", "sum"])
    
    va_oot_sub = train_fe.iloc[oot_va_idx]
    vc_oot = va_oot_sub[col].map(s_oot["count"]).fillna(0.0).values
    vs_oot = va_oot_sub[col].map(s_oot["sum"]).fillna(0.0).values
    vp_oot = va_oot_sub[prior_col].map(p_oot).fillna(global_mean).values
    oot_val_te_matrix[fname] = ((vs_oot + m * vp_oot) / (vc_oot + m)).astype(np.float32)
    
    # 2. Full 5-Fold GroupKFold OOF TE for Train & Test
    oof_te = np.full(len(train_fe), global_mean, dtype=np.float32)
    test_te_folds = np.zeros(len(test_fe), dtype=np.float32)
    
    for tr_idx, val_idx in folds:
        tr_sub = train_fe.iloc[tr_idx]
        prior_means = tr_sub.groupby(prior_col)["returned"].mean()
        stats = tr_sub.groupby(col)["returned"].agg(["count", "sum"])
        cnt_map = stats["count"]
        sum_map = stats["sum"]
        
        val_sub = train_fe.iloc[val_idx]
        v_cnt = val_sub[col].map(cnt_map).fillna(0.0).values
        v_sum = val_sub[col].map(sum_map).fillna(0.0).values
        v_prior = val_sub[prior_col].map(prior_means).fillna(global_mean).values
        oof_te[val_idx] = (v_sum + m * v_prior) / (v_cnt + m)
        
        t_cnt = test_fe[col].map(cnt_map).fillna(0.0).values
        t_sum = test_fe[col].map(sum_map).fillna(0.0).values
        t_prior = test_fe[prior_col].map(prior_means).fillna(global_mean).values
        test_te_folds += ((t_sum + m * t_prior) / (t_cnt + m)) / N_SPLITS
        
    train_fe[fname] = oof_te
    test_fe[fname] = test_te_folds

SAFE_CAT_COLS = [
    "tz_hint", "channel", "payment_method", "ship_mode", "promo_channel_code",
    "warehouse_zone", "picker_shift", "device_type", "category", "packaging_type",
    "color", "channel_store", "region", "format", "region_cust", "customer_type",
    "age_band", "preferred_contact", "loyalty_tier", "cust_country", "store_country",
    "temp_regime"
]

DROP_COLS = [
    "order_line_id", "order_id", "customer_id", "product_id", "store_id",
    "supplier_code", "coupon_code", "cust_cat_key", "prod_chan_key",
    "store_cat_key", "cat_retwin_key", "ship_zone_key", "cat_temp_key",
    "order_ts", "order_dt", "week_start", "is_test", "returned", "orig_idx",
    "unit_price", "order_value", "quantity"
]

train_fe.drop(columns=[c for c in DROP_COLS if c in train_fe.columns], inplace=True)
test_fe.drop(columns=[c for c in DROP_COLS if c in test_fe.columns], inplace=True)
gc.collect()

cat_dims = []
for col in SAFE_CAT_COLS:
    train_fe[col] = train_fe[col].astype(str)
    test_fe[col] = test_fe[col].astype(str)
    cats = sorted(list(set(train_fe[col].unique()).union(set(test_fe[col].unique()))))
    cat_map = {v: i for i, v in enumerate(cats)}
    train_fe[col] = train_fe[col].map(cat_map).fillna(0).astype(np.int16)
    test_fe[col] = test_fe[col].map(cat_map).fillna(0).astype(np.int16)
    cat_dims.append(len(cats) + 1)

FEATURE_COLS = list(train_fe.columns)
NUM_COLS = [c for c in FEATURE_COLS if c not in SAFE_CAT_COLS]

for col in NUM_COLS:
    train_fe[col] = train_fe[col].replace([np.inf, -np.inf], np.nan)
    test_fe[col] = test_fe[col].replace([np.inf, -np.inf], np.nan)
    med = float(train_fe[col].median())
    train_fe[col] = train_fe[col].fillna(med)
    test_fe[col] = test_fe[col].fillna(med)
    if train_fe[col].nunique() > 10:
        q_low, q_high = np.percentile(train_fe[col], [0.5, 99.5])
        train_fe[col] = np.clip(train_fe[col], q_low, q_high).astype(np.float32)
        test_fe[col] = np.clip(test_fe[col], q_low, q_high).astype(np.float32)
    else:
        train_fe[col] = train_fe[col].astype(np.float32)
        test_fe[col] = test_fe[col].astype(np.float32)

print(f"Total Modeling Features: {len(FEATURE_COLS)} | train_fe RAM: {train_fe.memory_usage().sum() / 1e6:.1f} MB")

# Build strict Out-of-Time Validation DataFrame (Months 20-24 with TE fitted ONLY on Months 1-19!)
X_oot_val = train_fe.iloc[oot_va_idx].copy()
for fname, vals in oot_val_te_matrix.items():
    X_oot_val[fname] = vals
y_oot_val = y_all[oot_va_idx]
del oot_val_te_matrix
gc.collect()

# ==============================================================================
# 5. STAGE A: OUT-OF-TIME FEATURE SELECTION & LIGHTGBM OPTUNA TUNING
# ==============================================================================
print("\n--- Stage A: Tuning LightGBM on Strict Out-of-Time Split (Months 1-19 -> Months 20-24) ---")
rng = np.random.default_rng(SEED)
sub_oot_tr = rng.choice(oot_tr_idx, size=min(250_000, len(oot_tr_idx)), replace=False)

X_opt_tr = train_fe.iloc[sub_oot_tr]
y_opt_tr = y_all[sub_oot_tr]
w_opt_tr = recency_weights[sub_oot_tr]

no_cust_te = [c for c in FEATURE_COLS if c not in ("customer_id_eb_te", "cust_cat_key_eb_te")]
no_all_te = [c for c in FEATURE_COLS if c not in te_col_names]

best_cols = FEATURE_COLS
best_probe_auc = 0.0
for name_set, cols_set in [
    ("Full Features + All Multi-Table TE", FEATURE_COLS),
    ("Full Features + Product/Store TE (No Customer TE)", no_cust_te),
    ("Pure Multi-Table Engineered Features (No TE)", no_all_te),
]:
    m_p = lgb.LGBMClassifier(
        n_estimators=600, learning_rate=0.04, num_leaves=45, max_depth=6,
        min_child_samples=250, subsample=0.8, colsample_bytree=0.65,
        reg_alpha=5.0, reg_lambda=15.0, verbose=-1, n_jobs=LGB_THREADS, random_state=SEED
    )
    m_p.fit(
        X_opt_tr[cols_set], y_opt_tr,
        sample_weight=w_opt_tr,
        eval_set=[(X_oot_val[cols_set], y_oot_val)],
        categorical_feature=[c for c in SAFE_CAT_COLS if c in cols_set],
        callbacks=[lgb.early_stopping(40, verbose=False)]
    )
    sc = roc_auc_score(y_oot_val, m_p.predict_proba(X_oot_val[cols_set])[:, 1])
    print(f"Strict Out-of-Time Probe [{name_set:50s}] -> Kaggle Proxy AUC: {sc:.5f}")
    if sc > best_probe_auc:
        best_probe_auc = sc
        best_cols = cols_set
    del m_p

FEATURE_COLS = best_cols
SAFE_CAT_COLS = [c for c in SAFE_CAT_COLS if c in FEATURE_COLS]
NUM_COLS = [c for c in FEATURE_COLS if c not in SAFE_CAT_COLS]

drop_extra = [c for c in train_fe.columns if c not in FEATURE_COLS]
if drop_extra:
    train_fe.drop(columns=drop_extra, inplace=True)
    test_fe.drop(columns=drop_extra, inplace=True)
    X_oot_val.drop(columns=drop_extra, inplace=True)
gc.collect()

X_opt_tr = train_fe.iloc[sub_oot_tr]

def lgb_objective(trial):
    params = {
        "objective": "binary",
        "metric": "auc",
        "boosting_type": "gbdt",
        "n_estimators": 1000,
        "learning_rate": trial.suggest_float("learning_rate", 0.02, 0.06, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 24, 80),
        "max_depth": trial.suggest_int("max_depth", 4, 8),
        "min_child_samples": trial.suggest_int("min_child_samples", 150, 700),
        "subsample": trial.suggest_float("subsample", 0.65, 0.90),
        "subsample_freq": 1,
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.45, 0.80),
        "reg_alpha": trial.suggest_float("reg_alpha", 1.0, 30.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 2.0, 40.0, log=True),
        "cat_smooth": trial.suggest_float("cat_smooth", 25.0, 100.0),
        "max_bin": 255,
        "verbose": -1,
        "n_jobs": LGB_THREADS,
        "random_state": SEED,
    }
    model = lgb.LGBMClassifier(**params)
    model.fit(
        X_opt_tr, y_opt_tr,
        sample_weight=w_opt_tr,
        eval_set=[(X_oot_val, y_oot_val)],
        categorical_feature=SAFE_CAT_COLS,
        callbacks=[lgb.early_stopping(40, verbose=False)]
    )
    preds = model.predict_proba(X_oot_val)[:, 1]
    return roc_auc_score(y_oot_val, preds)

study_lgb = optuna.create_study(direction="maximize")
study_lgb.optimize(lgb_objective, n_trials=10, timeout=150)
print(f"\n>>> LIGHTGBM STRICT OUT-OF-TIME VALIDATION AUC (KAGGLE LB PROXY): {study_lgb.best_value:.5f} <<<")
print("Best Tuned LightGBM Parameters:", study_lgb.best_params)

best_lgb_params = {
    **study_lgb.best_params,
    "objective": "binary",
    "metric": "auc",
    "boosting_type": "gbdt",
    "subsample_freq": 1,
    "max_bin": 255,
    "verbose": -1,
    "n_jobs": LGB_THREADS,
}

# ==============================================================================
# 6. DEEP LEARNING ARCHITECTURE (PYTORCH FAST TABULAR RESNET + OPTUNA TUNING)
#    & LINEAR / POLYNOMIAL / BAGGING MODELS
# ==============================================================================
print("\n--- Preparing Standardized Matrices & Entity Embeddings for Deep Learning & Linear Models ---")
sample_idx = rng.choice(len(train_fe), size=min(80_000, len(train_fe)), replace=False)
raw_corrs = np.abs([
    np.corrcoef(train_fe[c].values[sample_idx], y_all[sample_idx])[0, 1] for c in NUM_COLS
])
top55_idx = np.argsort(np.nan_to_num(raw_corrs))[::-1][:min(55, len(NUM_COLS))]
TOP_NUM_COLS = [NUM_COLS[i] for i in top55_idx]

X_tr_lin = train_fe[TOP_NUM_COLS].copy()
X_te_lin = test_fe[TOP_NUM_COLS].copy()
X_oot_lin = X_oot_val[TOP_NUM_COLS].copy()
del X_oot_val, X_opt_tr
gc.collect()

skew_vals = X_tr_lin.iloc[sample_idx].skew().abs()
skewed_cols = skew_vals[skew_vals > 2.0].index.tolist()[:15]

if len(skewed_cols) > 0:
    pt = PowerTransformer(method="yeo-johnson", standardize=False)
    pt.fit(X_tr_lin[skewed_cols].iloc[sample_idx])
    X_tr_lin[skewed_cols] = np.clip(pt.transform(X_tr_lin[skewed_cols]), -1e5, 1e5).astype(np.float32)
    X_te_lin[skewed_cols] = np.clip(pt.transform(X_te_lin[skewed_cols]), -1e5, 1e5).astype(np.float32)
    X_oot_lin[skewed_cols] = np.clip(pt.transform(X_oot_lin[skewed_cols]), -1e5, 1e5).astype(np.float32)

scaler = StandardScaler()
X_tr_std = np.nan_to_num(scaler.fit_transform(X_tr_lin), nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
X_te_std = np.nan_to_num(scaler.transform(X_te_lin), nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
X_oot_std = np.nan_to_num(scaler.transform(X_oot_lin), nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
del X_tr_lin, X_te_lin, X_oot_lin
gc.collect()

# Pre-calculate unified categorical embedding offsets for instant C++ embedding lookup
cat_offsets = np.cumsum([0] + cat_dims[:-1]).astype(np.int64)
total_cat_tokens = int(sum(cat_dims))
X_cat_tr = (train_fe[SAFE_CAT_COLS].values.astype(np.int64) + cat_offsets)
X_cat_te = (test_fe[SAFE_CAT_COLS].values.astype(np.int64) + cat_offsets)
X_cat_oot = X_cat_tr[oot_va_idx]

class FastTabularResNet(nn.Module):
    """
    Deep Learning Tabular Architecture:
    - Unified Vectorized Entity Embeddings across all categorical columns
    - Squeeze-and-Excitation Feature Attention Gate
    - Residual Highway Blocks with BatchNorm1d, SiLU (Swish), and Dropout
    """
    def __init__(self, num_features, total_tokens, n_cats, emb_dim=6, emb_drop=0.10, hidden_dim=128, dropout=0.25, n_blocks=2):
        super().__init__()
        self.unified_emb = nn.Embedding(total_tokens, emb_dim)
        self.emb_drop = nn.Dropout(emb_drop)
        
        in_dim = num_features + n_cats * emb_dim
        self.input_gate = nn.Sequential(
            nn.Linear(in_dim, in_dim),
            nn.Sigmoid()
        )
        self.input_layer = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.SiLU()
        )
        self.blocks = nn.ModuleList([
            nn.Sequential(
                nn.Linear(hidden_dim, hidden_dim),
                nn.BatchNorm1d(hidden_dim),
                nn.SiLU(),
                nn.Dropout(dropout),
                nn.Linear(hidden_dim, hidden_dim),
                nn.BatchNorm1d(hidden_dim)
            )
            for _ in range(n_blocks)
        ])
        self.act = nn.SiLU()
        self.head = nn.Linear(hidden_dim, 1)
        
    def forward(self, x_num, x_cat_offset):
        x_e = self.emb_drop(self.unified_emb(x_cat_offset).flatten(1))
        x = torch.cat([x_num, x_e], dim=1)
        x = x * self.input_gate(x)
        h = self.input_layer(x)
        for block in self.blocks:
            h = self.act(h + block(h))
        return self.head(h).squeeze(-1)

def pairwise_auc_bce_loss(logits, targets, sample_weights, rank_weight=0.25):
    bce = nn.functional.binary_cross_entropy_with_logits(logits, targets, weight=sample_weights)
    if rank_weight <= 0.0:
        return bce
    pos_mask = targets > 0.5
    neg_mask = ~pos_mask
    if pos_mask.any() and neg_mask.any():
        pos_l = logits[pos_mask][:256].unsqueeze(1)
        neg_l = logits[neg_mask][:256].unsqueeze(0)
        pair_loss = nn.functional.softplus(-(pos_l - neg_l)).mean()
        return (1.0 - rank_weight) * bce + rank_weight * pair_loss
    return bce

def eval_dl_model(model, X_num_np, X_cat_np, batch_size=16384):
    model.eval()
    preds = np.empty(len(X_num_np), dtype=np.float32)
    with torch.no_grad():
        for i in range(0, len(X_num_np), batch_size):
            end = min(i + batch_size, len(X_num_np))
            xn = torch.from_numpy(X_num_np[i:end]).to(DEVICE)
            xc = torch.from_numpy(X_cat_np[i:end]).to(DEVICE)
            preds[i:end] = torch.sigmoid(model(xn, xc)).cpu().numpy()
    return preds

def train_dl_epoch_loop(X_num_tr, X_cat_tr_sub, y_tr_sub, w_tr_sub, params, epochs=3, batch_size=8192, seed=SEED):
    torch.manual_seed(seed)
    model = FastTabularResNet(
        num_features=X_num_tr.shape[1],
        total_tokens=total_cat_tokens,
        n_cats=len(SAFE_CAT_COLS),
        emb_dim=params["emb_dim"],
        emb_drop=params["emb_drop"],
        hidden_dim=params["hidden_dim"],
        dropout=params["dropout"],
        n_blocks=params["n_blocks"]
    ).to(DEVICE)
    
    n_samples = len(X_num_tr)
    steps_per_epoch = max(1, (n_samples + batch_size - 1) // batch_size)
    optimizer = torch.optim.AdamW(model.parameters(), lr=params["lr"], weight_decay=params["weight_decay"])
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=params["lr"], total_steps=epochs * steps_per_epoch, pct_start=0.25
    )
    
    xn_t = torch.from_numpy(X_num_tr).to(DEVICE)
    xc_t = torch.from_numpy(X_cat_tr_sub).to(DEVICE)
    y_t = torch.from_numpy(y_tr_sub.astype(np.float32)).to(DEVICE)
    w_t = torch.from_numpy(w_tr_sub).to(DEVICE)
    
    model.train()
    for _ in range(epochs):
        perm = torch.randperm(n_samples, device=DEVICE)
        for i in range(0, n_samples, batch_size):
            idx = perm[i:i + batch_size]
            optimizer.zero_grad()
            logits = model(xn_t[idx], xc_t[idx])
            loss = pairwise_auc_bce_loss(logits, y_t[idx], w_t[idx], rank_weight=params["rank_weight"])
            loss.backward()
            optimizer.step()
            scheduler.step()
    del xn_t, xc_t, y_t, w_t
    return model

print("\n--- Tuning Deep Learning (FastTabularResNet) Hyperparameters via Optuna on Out-of-Time Holdout ---")
dl_sub_tr = rng.choice(oot_tr_idx, size=min(150_000, len(oot_tr_idx)), replace=False)
dl_sub_va = rng.choice(len(oot_va_idx), size=min(70_000, len(oot_va_idx)), replace=False)

def dl_objective(trial):
    dl_params = {
        "emb_dim": trial.suggest_categorical("emb_dim", [4, 6, 8]),
        "emb_drop": trial.suggest_float("emb_drop", 0.05, 0.25),
        "hidden_dim": trial.suggest_categorical("hidden_dim", [96, 128, 192]),
        "dropout": trial.suggest_float("dropout", 0.15, 0.40),
        "n_blocks": trial.suggest_int("n_blocks", 1, 2),
        "lr": trial.suggest_float("lr", 1.5e-3, 5e-3, log=True),
        "weight_decay": trial.suggest_float("weight_decay", 1e-5, 1e-3, log=True),
        "rank_weight": trial.suggest_float("rank_weight", 0.10, 0.40),
    }
    net = train_dl_epoch_loop(
        X_tr_std[dl_sub_tr], X_cat_tr[dl_sub_tr],
        y_all[dl_sub_tr], recency_weights[dl_sub_tr],
        dl_params, epochs=3, batch_size=8192, seed=SEED + trial.number
    )
    preds = eval_dl_model(net, X_oot_std[dl_sub_va], X_cat_oot[dl_sub_va])
    del net
    return roc_auc_score(y_oot_val[dl_sub_va], preds)

study_dl = optuna.create_study(direction="maximize")
study_dl.optimize(dl_objective, n_trials=6, timeout=110)
best_dl_params = study_dl.best_params
print(f"Best Deep Learning (FastTabularResNet) Out-of-Time Validation AUC: {study_dl.best_value:.5f}")
print("Best Deep Learning Hyperparameters:", best_dl_params)
del X_oot_std, X_cat_oot
gc.collect()

# ==============================================================================
# 7. 5-FOLD CROSS-VALIDATION FOR DEEP LEARNING, POLY ELASTICNET SGD & BAGGING
# ==============================================================================
print("\n--- Training 5-Fold CV for Deep Learning (FastTabularResNet), Poly ElasticNet SGD & Bagging ---")
oof_dl = np.zeros(len(train_fe), dtype=np.float64)
pred_dl = np.zeros(len(test_fe), dtype=np.float64)

oof_poly = np.zeros(len(train_fe), dtype=np.float64)
pred_poly = np.zeros(len(test_fe), dtype=np.float64)

oof_bag = np.zeros(len(train_fe), dtype=np.float64)
pred_bag = np.zeros(len(test_fe), dtype=np.float64)

poly = PolynomialFeatures(degree=2, include_bias=False)
X_tr_poly = np.hstack([X_tr_std, poly.fit_transform(X_tr_std[:, :10]).astype(np.float32)])
X_te_poly = np.hstack([X_te_std, poly.transform(X_te_std[:, :10]).astype(np.float32)])
n_poly_cols = X_tr_poly.shape[1]

for fold, (tr_idx, val_idx) in enumerate(folds):
    # 1. Deep Learning Model: Tuned FastTabularResNet
    dl_tr_rows = rng.choice(tr_idx, size=min(200_000, len(tr_idx)), replace=False)
    dl_model = train_dl_epoch_loop(
        X_tr_std[dl_tr_rows], X_cat_tr[dl_tr_rows],
        y_all[dl_tr_rows], recency_weights[dl_tr_rows],
        best_dl_params, epochs=4, batch_size=8192, seed=SEED + fold
    )
    oof_dl[val_idx] = eval_dl_model(dl_model, X_tr_std[val_idx], X_cat_tr[val_idx])
    pred_dl += eval_dl_model(dl_model, X_te_std, X_cat_te) / N_SPLITS
    del dl_model
    
    # 2. Polynomial ElasticNet SGD Logistic Regression
    sub_tr = rng.choice(tr_idx, size=min(250_000, len(tr_idx)), replace=False)
    m_poly = SGDClassifier(
        loss="log_loss", penalty="elasticnet", alpha=1.5e-3, l1_ratio=0.35,
        max_iter=25, tol=1e-3, random_state=SEED + fold
    )
    m_poly.fit(X_tr_poly[sub_tr], y_all[sub_tr], sample_weight=recency_weights[sub_tr])
    oof_poly[val_idx] = m_poly.predict_proba(X_tr_poly[val_idx])[:, 1]
    pred_poly += m_poly.predict_proba(X_te_poly)[:, 1] / N_SPLITS
    
    # 3. Zero-Multiprocessing In-Process Feature & Sample Bagging Ensemble
    n_bags = 8
    fold_bag_val = np.zeros(len(val_idx), dtype=np.float64)
    fold_bag_test = np.zeros(len(test_fe), dtype=np.float64)
    for b in range(n_bags):
        b_rows = rng.choice(tr_idx, size=min(100_000, len(tr_idx)), replace=False)
        b_cols = rng.choice(n_poly_cols, size=int(0.75 * n_poly_cols), replace=False)
        X_sub_bag = X_tr_poly[b_rows][:, b_cols]
        m_b = SGDClassifier(
            loss="log_loss", penalty="l2", alpha=1e-3,
            max_iter=20, tol=1e-3, random_state=SEED + fold * 100 + b
        )
        m_b.fit(X_sub_bag, y_all[b_rows], sample_weight=recency_weights[b_rows])
        fold_bag_val += m_b.predict_proba(X_tr_poly[val_idx][:, b_cols])[:, 1] / n_bags
        fold_bag_test += m_b.predict_proba(X_te_poly[:, b_cols])[:, 1] / n_bags
        del X_sub_bag
        
    oof_bag[val_idx] = fold_bag_val
    pred_bag += fold_bag_test / N_SPLITS

print(f"[Model 1: Deep Learning ResNet] 5-Fold GroupKFold OOF AUC: {roc_auc_score(y_all, oof_dl):.5f}")
print(f"[Model 2: Poly ElasticNet SGD]  5-Fold GroupKFold OOF AUC: {roc_auc_score(y_all, oof_poly):.5f}")
print(f"[Model 3: Bagging Logistic]     5-Fold GroupKFold OOF AUC: {roc_auc_score(y_all, oof_bag):.5f}")

del X_tr_std, X_te_std, X_cat_tr, X_cat_te, X_tr_poly, X_te_poly
gc.collect()

# ==============================================================================
# 8. 5-FOLD GROUPKFOLD(ORDER_ID) FOR LIGHTGBM (GBDT + EXTRATREES)
# ==============================================================================
print("\n--- Training 5-Fold GroupKFold LightGBM (GBDT + ExtraTrees across 100% of timeline) ---")
oof_lgb = np.zeros(len(train_fe), dtype=np.float64)
pred_lgb = np.zeros(len(test_fe), dtype=np.float64)

oof_lgb_et = np.zeros(len(train_fe), dtype=np.float64)
pred_lgb_et = np.zeros(len(test_fe), dtype=np.float64)

et_params = {
    **best_lgb_params,
    "extra_trees": True,
    "colsample_bytree": min(0.60, best_lgb_params.get("colsample_bytree", 0.60)),
    "min_child_samples": max(280, best_lgb_params.get("min_child_samples", 280)),
}

for fold, (tr_idx, val_idx) in enumerate(folds):
    X_tr_f, y_tr_f, w_tr_f = train_fe.iloc[tr_idx], y_all[tr_idx], recency_weights[tr_idx]
    X_va_f, y_va_f = train_fe.iloc[val_idx], y_all[val_idx]
    
    # Model 4A: Tuned Standard LightGBM GBDT
    model_gbdt = lgb.LGBMClassifier(**best_lgb_params, n_estimators=2000, random_state=SEED + fold)
    model_gbdt.fit(
        X_tr_f, y_tr_f,
        sample_weight=w_tr_f,
        eval_set=[(X_va_f, y_va_f)],
        categorical_feature=SAFE_CAT_COLS,
        callbacks=[lgb.early_stopping(50, verbose=False)]
    )
    oof_lgb[val_idx] = model_gbdt.predict_proba(X_va_f)[:, 1]
    pred_lgb += model_gbdt.predict_proba(test_fe)[:, 1] / N_SPLITS
    del model_gbdt
    
    # Model 4B: Randomized ExtraTrees LightGBM
    model_et = lgb.LGBMClassifier(**et_params, n_estimators=2000, random_state=SEED + 100 + fold)
    model_et.fit(
        X_tr_f, y_tr_f,
        sample_weight=w_tr_f,
        eval_set=[(X_va_f, y_va_f)],
        categorical_feature=SAFE_CAT_COLS,
        callbacks=[lgb.early_stopping(50, verbose=False)]
    )
    oof_lgb_et[val_idx] = model_et.predict_proba(X_va_f)[:, 1]
    pred_lgb_et += model_et.predict_proba(test_fe)[:, 1] / N_SPLITS
    del model_et, X_tr_f, X_va_f
    gc.collect()
    
    f_auc_gbdt = roc_auc_score(y_va_f, oof_lgb[val_idx])
    f_auc_et = roc_auc_score(y_va_f, oof_lgb_et[val_idx])
    print(f"Fold {fold+1}/{N_SPLITS} | LGB-GBDT AUC: {f_auc_gbdt:.5f} | LGB-ExtraTrees AUC: {f_auc_et:.5f} | Elapsed: {(time.time() - t_start)/60:.1f}m")

print(f"\n[Model 4A: LightGBM GBDT]       5-Fold GroupKFold OOF AUC: {roc_auc_score(y_all, oof_lgb):.5f}")
print(f"[Model 4B: LightGBM ExtraTrees] 5-Fold GroupKFold OOF AUC: {roc_auc_score(y_all, oof_lgb_et):.5f}")

del train_fe, test_fe
gc.collect()

# ==============================================================================
# 9. HYBRID NEURAL + GBDT + LINEAR META-ENSEMBLE & BASKET POST-PROCESSING
# ==============================================================================
print("\n--- Optimizing Cross-Validated Neural + GBDT + Linear Meta-Ensemble Weights ---")
oof_dict = {
    "LGB_GBDT": oof_lgb,
    "LGB_ExtraTrees": oof_lgb_et,
    "Deep_TabularResNet": oof_dl,
    "Poly_ElasticNet": oof_poly,
    "Bagging_LR": oof_bag,
}
pred_dict = {
    "LGB_GBDT": pred_lgb,
    "LGB_ExtraTrees": pred_lgb_et,
    "Deep_TabularResNet": pred_dl,
    "Poly_ElasticNet": pred_poly,
    "Bagging_LR": pred_bag,
}
names = list(oof_dict.keys())

oof_ranks = np.column_stack([pd.Series(oof_dict[k]).rank(pct=True).values for k in names])
pred_ranks = np.column_stack([pd.Series(pred_dict[k]).rank(pct=True).values for k in names])

def ens_objective(trial):
    w = np.array([trial.suggest_float(f"w_{k}", 0.0, 1.0) for k in names])
    if w.sum() == 0:
        return 0.0
    w /= w.sum()
    return roc_auc_score(y_all, np.dot(oof_ranks, w))

ens_study = optuna.create_study(direction="maximize")
ens_study.optimize(ens_objective, n_trials=300)

w_opt = np.array([ens_study.best_params[f"w_{k}"] for k in names])
w_opt /= w_opt.sum()

for k, weight in zip(names, w_opt):
    print(f"Ensemble Weight [{k:18s}]: {weight:.4f} | 5-Fold OOF AUC: {roc_auc_score(y_all, oof_dict[k]):.5f}")

raw_oof_ens = np.dot(oof_ranks, w_opt)
raw_test_ens = np.dot(pred_ranks, w_opt)

# Intra-Order Basket Risk Halo Smoothing
best_alpha = 0.0
best_pp_auc = roc_auc_score(y_all, raw_oof_ens)
ord_mean_oof = pd.DataFrame({"order_id": train_order_id, "pred": raw_oof_ens}).groupby("order_id", sort=False)["pred"].transform("mean").values

for alpha in [0.02, 0.04, 0.06, 0.08, 0.10, 0.12]:
    pp_oof = (1.0 - alpha) * raw_oof_ens + alpha * ord_mean_oof
    score = roc_auc_score(y_all, pp_oof)
    if score > best_pp_auc:
        best_pp_auc = score
        best_alpha = alpha

if best_alpha > 0:
    ord_mean_test = pd.DataFrame({"order_id": test_order_id, "pred": raw_test_ens}).groupby("order_id", sort=False)["pred"].transform("mean").values
    final_oof = (1.0 - best_alpha) * raw_oof_ens + best_alpha * ord_mean_oof
    final_test = (1.0 - best_alpha) * raw_test_ens + best_alpha * ord_mean_test
else:
    final_oof = raw_oof_ens
    final_test = raw_test_ens

print(f"\n======================================================================")
print(f">>> STRICT OUT-OF-TIME HOLDOUT AUC (KAGGLE LB PROXY): {study_lgb.best_value:.5f} <<<")
print(f">>> FULL 5-FOLD GROUPKFOLD ENSEMBLE OOF ROC-AUC:      {roc_auc_score(y_all, final_oof):.5f} <<<")
print(f"======================================================================")

# ==============================================================================
# 10. VERIFIED KAGGLE SUBMISSION GENERATION
# ==============================================================================
submission = pd.DataFrame({
    "order_line_id": test_order_line_id,
    "returned": np.clip(final_test, 0.0, 1.0)
})
submission = sub_df[["order_line_id"]].merge(submission, on="order_line_id", how="left")
assert len(submission) == 479995, f"Expected 479,995 rows, got {len(submission)}"
assert submission["returned"].isna().sum() == 0, "NaN values detected in submission!"

submission.to_csv("submission10.csv", index=False)
print(f"\nSaved 'submission10.csv' successfully in {(time.time() - t_start)/60:.2f} minutes!")
print(submission.describe())

c:\python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Running on Device: cpu | CPU Threads: 8
Loaded train.csv: (1400005, 27) | test.csv: (479995, 26)
Executing deep multi-table null & outlier reconstruction across all 5 datasets...
Multi-table reconstruction & feature engineering completed in 35.8s!

--- Computing Leak-Free Out-of-Time TE (for Local LB Alignment) & 5-Fold OOF TE ---
Total Modeling Features: 136 | train_fe RAM: 700.0 MB

--- Stage A: Tuning LightGBM on Strict Out-of-Time Split (Months 1-19 -> Months 20-24) ---
Strict Out-of-Time Probe [Full Features + All Multi-Table TE                ] -> Kaggle Proxy AUC: 0.65559
Strict Out-of-Time Probe [Full Features + Product/Store TE (No Customer TE) ] -> Kaggle Proxy AUC: 0.65120
Strict Out-of-Time Probe [Pure Multi-Table Engineered Features (No TE)      ] -> Kaggle Proxy AUC: 0.62922

>>> LIGHTGBM STRICT OUT-OF-TIME VALIDATION AUC (KAGGLE LB PROXY): 0.66775 <<<
Best Tuned LightGBM Parameters: {'learning_rate': 0.03519488613909274, 'num_leaves': 75, 'max_depth': 8, 'min_child_sampl